In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

kernel = "rbf"
nu_list = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
gamma_list = [1.0]

data_path = Path("../data_processed/NASA/nasa_battery_features_rolling.csv")
results_root = Path("../results/OCSVM_NASA")
results_root.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path)

feature_cols = [
    "V", "I", "T",
    "Current_load", "Voltage_load", "Time",
    "V_diff1", "I_diff1", "T_diff1",
    "V_roll_mean", "V_roll_std", "V_roll_min", "V_roll_max",
    "I_roll_mean", "I_roll_std",
    "T_roll_mean", "T_roll_std"
]

required_cols = feature_cols + ["y_true_file"]
df = df.dropna(subset=required_cols).reset_index(drop=True)

X = df[feature_cols].copy()
y_true = df["y_true_file"].astype(int)

X_normal = X[y_true == 0].copy()
X_anomaly = X[y_true == 1].copy()

scaler = StandardScaler()

max_train_samples = 5000

if len(X_normal) > max_train_samples:
    X_train_raw = X_normal.sample(n=max_train_samples, random_state=42)
else:
    X_train_raw = X_normal.copy()

X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=feature_cols
)

X_test_normal = X_normal.copy()
y_test_normal = pd.Series(np.zeros(len(X_test_normal), dtype=int))

n_anomaly_test = min(2000, len(X_anomaly))
X_test_anomaly = X_anomaly.sample(n=n_anomaly_test, random_state=42)
y_test_anomaly = pd.Series(np.ones(len(X_test_anomaly), dtype=int))

X_test = pd.concat([X_test_normal, X_test_anomaly], axis=0)
y_test = pd.concat([y_test_normal, y_test_anomaly], axis=0)

test_df = X_test.copy()
test_df["y_true"] = y_test.values
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

X_test = test_df.drop(columns=["y_true"])
y_test = test_df["y_true"].astype(int)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols
)

def gamma_to_str(gamma):
    if isinstance(gamma, str):
        return gamma
    return str(gamma).replace(".", "_")

def nu_to_str(nu):
    return str(nu).replace(".", "_")

all_results = []

for gamma in gamma_list:
    gamma_dir = results_root / f"gamma_{gamma_to_str(gamma)}"
    gamma_dir.mkdir(parents=True, exist_ok=True)

    gamma_results = []

    for nu in nu_list:
        exp_name = f"kernel_{kernel}_nu_{nu_to_str(nu)}"
        exp_dir = gamma_dir / exp_name
        exp_dir.mkdir(parents=True, exist_ok=True)

        model = OneClassSVM(
            kernel=kernel,
            nu=nu,
            gamma=gamma
        )

        model.fit(X_train)

        pred_raw = model.predict(X_test_scaled)
        decision_scores = model.decision_function(X_test_scaled)
        score_samples = model.score_samples(X_test_scaled)

        df_results = X_test.copy()
        df_results["y_true"] = y_test.values
        df_results["ocsvm_pred_raw"] = pred_raw
        df_results["ocsvm_decision"] = decision_scores
        df_results["ocsvm_score"] = score_samples
        df_results["is_anomaly"] = (df_results["ocsvm_pred_raw"] == -1).astype(int)

        y_pred = df_results["is_anomaly"]

        cm = confusion_matrix(y_test, y_pred)

        accuracy = accuracy_score(y_test, y_pred) * 100
        precision = precision_score(y_test, y_pred, zero_division=0) * 100
        recall = recall_score(y_test, y_pred, zero_division=0) * 100
        f1 = f1_score(y_test, y_pred, zero_division=0) * 100

        result_row = {
            "experiment": exp_name,
            "kernel": kernel,
            "nu": nu,
            "gamma": gamma,
            "accuracy_%": round(accuracy, 2),
            "precision_%": round(precision, 2),
            "recall_%": round(recall, 2),
            "f1_%": round(f1, 2)
        }

        all_results.append(result_row)
        gamma_results.append(result_row)

        df_results.to_csv(exp_dir / "predictions.csv", index=False)

        top_anomalies = df_results.sort_values("ocsvm_score").head(20)
        top_anomalies.to_csv(exp_dir / "top20.csv", index=False)

        metrics_df = pd.DataFrame({
            "Metrika": [
                "Presnosť (Accuracy)",
                "Precíznosť (Precision)",
                "Citlivosť (Recall)",
                "F1-skóre"
            ],
            "Hodnota (%)": [
                round(accuracy, 2),
                round(precision, 2),
                round(recall, 2),
                round(f1, 2)
            ]
        })

        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")

        table = ax.table(
            cellText=metrics_df.values,
            colLabels=metrics_df.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)

        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_text_props(weight="bold")
                cell.set_facecolor("#dce6f1")
            else:
                cell.set_facecolor("#f9f9f9")

        plt.title("Výsledné metriky modelu One-Class SVM – NASA", fontsize=12, pad=20)
        plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title("Matica zámen (Confusion Matrix)")
        plt.colorbar()

        classes = ["Normálne", "Anomálie"]
        tick_marks = np.arange(len(classes))

        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)

        threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "black" if cm[i, j] > threshold else "white"
                plt.text(
                    j, i, cm[i, j],
                    ha="center", va="center",
                    color=color
                )

        plt.xlabel("Predikované triedy")
        plt.ylabel("Skutočné triedy")
        plt.tight_layout()
        plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        x_axis = np.arange(len(df_results))

        plt.figure(figsize=(14, 5))
        plt.plot(x_axis, df_results["V"], linewidth=1)

        anomaly_idx = df_results["is_anomaly"] == 1
        plt.scatter(
            np.array(x_axis)[anomaly_idx],
            df_results.loc[anomaly_idx, "V"],
            s=18
        )

        plt.title(f"{exp_name}, gamma={gamma_to_str(gamma)}")
        plt.xlabel("Index")
        plt.ylabel("V")
        plt.tight_layout()
        plt.savefig(exp_dir / "anomaly_plot.png", dpi=300, bbox_inches="tight")
        plt.close()

    gamma_results_df = pd.DataFrame(gamma_results)
    gamma_results_df = gamma_results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
    gamma_results_df.to_csv(gamma_dir / "summary_results.csv", index=False)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Done
